#### Purpose

Extract and compare the homeostatic metabolite targets from the FBA listener in the metabolic engineering sim to the biomass reaction of the iJO1366 model. This is motivated by the fact that the FBA listener's homeostatic targets are meant to reflect the cell's biomass needs, so we should see significant overlap with the biomass reaction components.

In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly.graph_objects import Scatter, Figure

from ecoli.processes.metabolism_redux import NetworkFlowModel, FlowResult
os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:

# import reaction info from sim result
time = '600'
date = '2026-04-17'
experiment = "metabolism_redux_metabolic_engineering"
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/metabolic_engineering/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output = output['agents']['0']
fba = output['listeners']['fba_results']
bulk = pd.DataFrame(output['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

In [3]:
# get commonly stored variables
metabolism = agent['ecoli-metabolism-redux']
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
fba_reaction_ids_to_base_reaction_ids = metabolism.parameters['fba_reaction_ids_to_base_reaction_ids']
metabolites = metabolism.metabolite_names.copy()
exchange_molecules = metabolism.exchange_molecules
kinetic_enzymes = metabolism.parameters['kinetic_constraint_enzymes']

In [4]:
fba.keys()

dict_keys(['media_id', 'conc_updates', 'base_reaction_fluxes', 'solution_fluxes', 'solution_dmdt', 'time_per_step', 'estimated_fluxes', 'estimated_homeostatic_dmdt', 'target_homeostatic_dmdt', 'target_homeostatic_dmdt_conc', 'estimated_homeostatic_dmdt_conc_raw', 'estimated_homeostatic_dmdt_conc', 'estimated_exchange_dmdt', 'estimated_intermediate_dmdt', 'target_kinetic_fluxes', 'target_kinetic_bounds', 'reaction_catalyst_counts', 'maintenance_target'])

In [5]:
homeostatic_ids = list(metabolism.homeostatic_metabolites)
assert homeostatic_ids == list(
  np.array(metabolism.network_flow_model.mets)[
      metabolism.network_flow_model.homeostatic_idx
  ]
)

target_homeostatic_counts = pd.DataFrame(fba["target_homeostatic_dmdt"],
                              columns=metabolism.homeostatic_metabolites).mean(axis=0)
estimated_homeostatic_counts = pd.DataFrame(fba["estimated_homeostatic_dmdt"],
                                    columns=metabolism.homeostatic_metabolites).mean(axis=0)
target_homeostatic_conc = pd.DataFrame(fba["target_homeostatic_dmdt_conc"],
                              columns=metabolism.homeostatic_metabolites).mean(axis=0)
estimated_homeostatic_conc = pd.DataFrame(fba["estimated_homeostatic_dmdt_conc"],
                                    columns=metabolism.homeostatic_metabolites).mean(axis=0)
estimated_homeostatic_conc_raw = pd.DataFrame(fba["estimated_homeostatic_dmdt_conc_raw"],
                                    columns=metabolism.homeostatic_metabolites).mean(axis=0)


In [6]:
estimated_homeostatic_conc

2-3-DIHYDROXYBENZOATE[c]     0.000035
2-KETOGLUTARATE[c]           0.000090
2-PG[c]                      0.000023
2K-4CH3-PENTANOATE[c]        0.000035
4-AMINO-BUTYRATE[c]          0.000078
                              ...    
WATER[c]                    10.980466
XYLULOSE-5-PHOSPHATE[c]      0.000043
ZN+2[c]                      0.000019
ZN+2[p]                      0.000026
glycogen-monomer[c]          0.008263
Length: 172, dtype: float64

#### FBA - Core


In [7]:
import requests
import pandas as pd
from cobra.io import load_json_model
import time

model = load_json_model("notebooks/metabolic_engineering/iML1515.json")
rxn = model.reactions.get_by_id("BIOMASS_Ec_iML1515_core_75p37M")

rows = []
for met, coeff in rxn.metabolites.items():
    # Strip compartment suffix for the API call (e.g. ala__L_c -> ala__L)
    universal_id = met.id.rsplit("_", 1)[0]

    ecocyc_id = None
    try:
        r = requests.get(
            f"http://bigg.ucsd.edu/api/v2/universal/metabolites/{universal_id}",
            timeout=5
        )
        if r.ok:
            links = r.json().get("database_links", {})
            # BiGG stores EcoCyc under 'BioCyc' key
            biocyc = links.get("BioCyc", [])
            ecocyc_id = biocyc[0]["id"] if biocyc else None
    except Exception:
        pass

    rows.append({
        "bigg_id": met.id,
        "name": met.name,
        "coefficient": coeff,
        "ecocyc_id": ecocyc_id
    })
    time.sleep(0.1)  # (~10 req/sec limit)

df = pd.DataFrame(rows).sort_values("coefficient")

In [8]:
df

,bigg_id,name,coefficient,ecocyc_id
10,atp_c,ATP C10H12N5O13P3,-75.552230,META:ATP
30,h2o_c,H2O H2O,-70.028756,META:CPD-15815
28,gly_c,Glycine,-0.612638,META:GLY
5,ala__L_c,L-Alanine,-0.513689,META:L-ALPHA-ALANINE
36,leu__L_c,L-Leucine,-0.450531,META:LEU
...,...,...,...,...
11,btn_c,Biotin,-0.000002,META:BIOTIN
53,ppi_c,Diphosphate,0.773903,META:PPI
52,pi_c,Phosphate,75.373230,META:CPD-16459
4,adp_c,ADP C10H12N5O10P2,75.377230,META:ADP


In [9]:
COMPARTMENT_MAP = {"c": "[c]", "p": "[p]", "e": "[p]"}  # vEcoli uses [p] for periplasmic uptake

# Some BiGG entries use MetaCyc generic IDs where EcoCyc has a species-specific frame ID.
# Extend as you spot mismatches.
METACYC_TO_ECOCYC = {
  "CPD-15815": "WATER",
  "CPD-16459": "Pi",
  "CPD-3": "AMMONIUM",  # nh4_c commonly
  "CPD0-1958": "L-ORNITHINE",
}

def to_vecoli_id(row):
  if not row["ecocyc_id"]:
      return None
  frame = row["ecocyc_id"].removeprefix("META:")
  frame = METACYC_TO_ECOCYC.get(frame, frame)
  comp_letter = row["bigg_id"].rsplit("_", 1)[-1]
  return f"{frame}{COMPARTMENT_MAP.get(comp_letter, '[c]')}"

df["vecoli_id"] = df.apply(to_vecoli_id, axis=1)
df["in_homeostatic"] = df["vecoli_id"].isin(metabolism.homeostatic_metabolites)

homeostatic_set = set(metabolism.homeostatic_metabolites)
df["in_homeostatic"] = df["vecoli_id"].isin(homeostatic_set)
print(f"matched {df['in_homeostatic'].sum()} / {len(df)}")


matched 56 / 70


In [10]:
df[["bigg_id", "ecocyc_id", "vecoli_id", "in_homeostatic"]]

,bigg_id,ecocyc_id,vecoli_id,in_homeostatic
10,atp_c,META:ATP,ATP[c],True
30,h2o_c,META:CPD-15815,WATER[c],True
28,gly_c,META:GLY,GLY[c],True
5,ala__L_c,META:L-ALPHA-ALANINE,L-ALPHA-ALANINE[c],True
36,leu__L_c,META:LEU,LEU[c],True
...,...,...,...,...
11,btn_c,META:BIOTIN,BIOTIN[c],True
53,ppi_c,META:PPI,PPI[c],True
52,pi_c,META:CPD-16459,Pi[c],True
4,adp_c,META:ADP,ADP[c],True


In [11]:
frame_to_full = {}
for h in metabolism.homeostatic_metabolites:
  frame_to_full.setdefault(h.rsplit("[", 1)[0], []).append(h)

misses = df[~df["in_homeostatic"] & df["ecocyc_id"].notna()]
for _, r in misses.iterrows():
  frame = r["ecocyc_id"].removeprefix("META:")
  print(f"{r['bigg_id']:<15} -> {frame:<25} candidates: {frame_to_full.get(frame, [])}")

pe161_p         -> CPD-17086                 candidates: []
pe160_p         -> CPD-12819                 candidates: ['CPD-12819[c]']
kdo2lipid4_e    -> KDO2-LIPID-IVA            candidates: []
murein5px4p_p   -> CPD0-2278                 candidates: []
nh4_c           -> AMMONIA                   candidates: []
fe3_c           -> FE+3                      candidates: ['FE+3[p]']
so4_c           -> HSO4                      candidates: []
cu2_c           -> CU+2                      candidates: []
4fe4s_c         -> CPD-7                     candidates: []
2ohph_c         -> 2-OCTAPRENYL-6-HYDROXYPHENOL candidates: []
10fthf_c        -> 10-FORMYL-THF             candidates: []
udcpdp_c        -> CPD-9649                  candidates: []
mobd_c          -> CPD-3                     candidates: []


In [12]:
fba_ids = set(df["vecoli_id"].dropna())
homeo_ids = set(metabolism.homeostatic_metabolites)

in_both      = fba_ids & homeo_ids
only_in_fba  = fba_ids - homeo_ids
only_in_homeo = homeo_ids - fba_ids
unmapped_from_fba = df[df["vecoli_id"].isna()]["bigg_id"].tolist()

print(f"in both: {len(in_both):>4}")
print(f"only in fba: {len(only_in_fba):>4}")
print(f"only in homeo: {len(only_in_homeo):>4}")
print(f"unmapped (no ecocyc_id from BiGG): {len(unmapped_from_fba)}")

print("\nBoth:")
print(sorted(in_both))
print("\nFBA:")
print(sorted(only_in_fba))
print("\nvEcoli homeostatic:")
print(sorted(only_in_homeo))
print("\nFBA unmapped (no ecocyc_id from BiGG):")
print(unmapped_from_fba)

in both:   56
only in fba:   13
only in homeo:  116
unmapped (no ecocyc_id from BiGG): 1

Both:
['ADP[c]', 'ARG[c]', 'ASN[c]', 'ATP[c]', 'BIOTIN[c]', 'CA+2[c]', 'CL-[c]', 'CO+2[c]', 'CO-A[c]', 'CTP[c]', 'CYS[c]', 'DATP[c]', 'DCTP[c]', 'DGTP[c]', 'FAD[c]', 'FE+2[c]', 'GLN[c]', 'GLT[c]', 'GLY[c]', 'GTP[c]', 'HIS[c]', 'ILE[c]', 'K+[c]', 'L-ALPHA-ALANINE[c]', 'L-ASPARTATE[c]', 'LEU[c]', 'LYS[c]', 'METHYLENE-THF[c]', 'MET[c]', 'MG+2[c]', 'MN+2[c]', 'NADP[c]', 'NAD[c]', 'NI+2[c]', 'PHE[c]', 'PPI[c]', 'PROTOHEME[c]', 'PROTON[c]', 'PRO[c]', 'PYRIDOXAL_PHOSPHATE[c]', 'Pi[c]', 'RIBOFLAVIN[c]', 'S-ADENOSYLMETHIONINE[c]', 'SER[c]', 'SIROHEME[c]', 'SUC-COA[c]', 'THF[c]', 'THIAMINE-PYROPHOSPHATE[c]', 'THR[c]', 'TRP[c]', 'TTP[c]', 'TYR[c]', 'UTP[c]', 'VAL[c]', 'WATER[c]', 'ZN+2[c]']

FBA:
['10-FORMYL-THF[c]', '2-OCTAPRENYL-6-HYDROXYPHENOL[c]', 'AMMONIA[c]', 'AMMONIUM[c]', 'CPD-12819[p]', 'CPD-17086[p]', 'CPD-7[c]', 'CPD-9649[c]', 'CPD0-2278[p]', 'CU+2[c]', 'FE+3[c]', 'HSO4[c]', 'KDO2-LIPID-IVA[p]']

